# S6E8 V7: OOF Diversity Stack

## A serious attempt to move beyond the public plateau

This notebook does not promise to beat the current public leader. It builds the
experiment that must succeed before such a claim is credible: a self-trained
stack whose members are both strong and measurably different.

V7 reuses aligned V5 and V6 out-of-fold artifacts, then trains two new families:

1. **RealMLP**, a modern neural tabular model trained without target encodings;
2. **CatBoost lattice**, which sees raw numerical values and target-free exact-value
   categorical copies, allowing ordered target statistics to rediscover the
   synthetic value grid.

The final stack is selected with nested meta-validation, correlation clustering,
strong L2 regularization, fold-level stability checks, and a frozen acceptance
rule. The public leaderboard is used only after the files are produced.


## Research contract

- `id` is never a predictive feature.
- Every base prediction used for meta-training is out of fold.
- RealMLP receives no target encoding, public predictions, or test labels.
- CatBoost exact-value categories are built from feature frequencies only.
- Members with OOF Spearman correlation above `0.985` are clustered before the
  logistic meta-model is fitted.
- The L2 value is selected inside each held-out meta fold.
- V7 is accepted only if it improves the V5 OOF reference by more than the stated
  noise floor and remains stable across folds.
- Checkpoints are signed by the data and configuration, then written atomically to
  Google Drive.
- Anchor blending is disabled by default. No public-leaderboard weight search is
  performed here.


## 1. Install dependencies

RealMLP comes from the official `pytabkit` package. CatBoost supplies the second
new family. Colab already includes CUDA PyTorch, NumPy, Pandas, SciPy, and
scikit-learn.


In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "kaggle>=1.7",
        "catboost>=1.2.8",
        "pytabkit==1.7.3",
    ],
    check=True,
)
print("dependency install finished")


## 2. Imports and frozen configuration

`SMOKE_TEST=True` validates the mechanics on a small sample but never creates an
official-looking submission. The default full run uses the same five folds as V5.


In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from catboost import CatBoostClassifier, CatBoostError, Pool
from IPython.display import display
from pytabkit import RealMLP_TD_Classifier
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from scipy.stats import norm, rankdata, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.7f}")

PIPELINE_VERSION = "v7.0.0"
COMPETITION = "playground-series-s6e8"
TARGET = "addicted_label"
ID_COL = "id"

FULL_RUN = True
SMOKE_TEST = False
USE_GOOGLE_DRIVE = True
RUN_REALMLP = True
RUN_CATBOOST_LATTICE = True

EXPECTED_TRAIN_ROWS = 691_369
EXPECTED_TEST_ROWS = 296_302
FOLD_SEED = 42
N_SPLITS = 3 if SMOKE_TEST else 5

REALMLP_EPOCHS = 6 if SMOKE_TEST else 64
REALMLP_BATCH_SIZE = 2048
REALMLP_VAL_FRACTION = 0.15

CATBOOST_MAX_ITERATIONS = 250 if SMOKE_TEST else 2500
CATBOOST_EARLY_STOPPING = 30 if SMOKE_TEST else 120
CATBOOST_INNER_VALID_FRACTION = 0.10
CATBOOST_RARE_MIN_COUNT = 50

CORRELATION_CLUSTER_THRESHOLD = 0.985
META_C_GRID = [0.03, 0.10, 0.30, 1.00]
MIN_OVERALL_GAIN = 0.00005
MIN_FOLD_WINS = 2 if SMOKE_TEST else 4
MAX_WORST_FOLD_LOSS = 0.00003

ANCHOR_PUBLIC_AUC = 0.97128
CURRENT_PUBLIC_TARGET = 0.97199  # dated reference, never used for selection
OOF_TO_PUBLIC_PRIOR = 0.00102    # median of observed V5 and E1 offsets

RAW_NUMERIC = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time",
]
RAW_CATEGORICAL = [
    "gender",
    "stress_level",
    "academic_work_impact",
]
RAW_COLUMNS = RAW_NUMERIC + RAW_CATEGORICAL

V5_CANDIDATE_ALLOWLIST = [
    "best_single",
    "rank_average",
    "dual_logistic",
    "dual_regime_precommitted",
    "greedy_rank_nested",
    "greedy_regime_nested",
]

GPU_ACTIVE = torch.cuda.is_available()
DEVICE = "cuda" if GPU_ACTIVE else "cpu"
PYTABKIT_VERSION = importlib.metadata.version("pytabkit")
CATBOOST_VERSION = importlib.metadata.version("catboost")

CONFIG = {
    "pipeline": PIPELINE_VERSION,
    "smoke_test": SMOKE_TEST,
    "fold_seed": FOLD_SEED,
    "n_splits": N_SPLITS,
    "realmlp_epochs": REALMLP_EPOCHS,
    "realmlp_batch_size": REALMLP_BATCH_SIZE,
    "catboost_iterations": CATBOOST_MAX_ITERATIONS,
    "catboost_rare_min_count": CATBOOST_RARE_MIN_COUNT,
    "correlation_threshold": CORRELATION_CLUSTER_THRESHOLD,
    "meta_c_grid": META_C_GRID,
    "device": DEVICE,
    "pytabkit": PYTABKIT_VERSION,
    "catboost": CATBOOST_VERSION,
}
CONFIG_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:12]

print(f"pipeline            {PIPELINE_VERSION}")
print(f"signature           {CONFIG_SIGNATURE}")
print(f"GPU active          {GPU_ACTIVE}")
print(f"device              {DEVICE}")
print(f"PyTabKit            {PYTABKIT_VERSION}")
print(f"CatBoost            {CATBOOST_VERSION}")
print(f"folds               {N_SPLITS}, seed={FOLD_SEED}")
print(f"smoke test          {SMOKE_TEST}")

if not GPU_ACTIVE and not SMOKE_TEST:
    raise RuntimeError("Select a T4 GPU before starting the V7 full run.")


## 3. Persistent storage and secure Kaggle authentication

Create a Colab Secret named `KAGGLE_API_TOKEN` and enable notebook access. Its
value is never printed or saved by this notebook.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = (not IN_COLAB) and Path("/kaggle/input").exists()

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PERSIST_ROOT = Path("/content/drive/MyDrive/s6e8_v7")
elif IN_KAGGLE:
    PERSIST_ROOT = Path("/kaggle/working/s6e8_v7")
else:
    PERSIST_ROOT = Path("./s6e8_v7")

CHECKPOINT_DIR = PERSIST_ROOT / "checkpoints" / CONFIG_SIGNATURE
ARTIFACT_DIR = PERSIST_ROOT / "artifacts" / CONFIG_SIGNATURE
SUBMISSION_DIR = PERSIST_ROOT / "submissions" / CONFIG_SIGNATURE
for directory in (CHECKPOINT_DIR, ARTIFACT_DIR, SUBMISSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    from google.colab import userdata

    try:
        kaggle_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Create the KAGGLE_API_TOKEN Colab Secret and enable notebook access."
        ) from exc
    if not kaggle_token:
        raise RuntimeError("The KAGGLE_API_TOKEN secret is empty.")
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    del kaggle_token

print("persistent root  ", PERSIST_ROOT)
print("checkpoints      ", CHECKPOINT_DIR)
print("artifacts        ", ARTIFACT_DIR)
print("submissions      ", SUBMISSION_DIR)


## 4. Download and validate the competition data

The row counts, columns, IDs, target values, and submission order are checked
before any expensive work starts.


In [ ]:
if IN_KAGGLE:
    DATA_DIR = Path(f"/kaggle/input/{COMPETITION}")
else:
    DATA_DIR = Path("/content/s6e8_data") if IN_COLAB else Path("./data")
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    required = ["train.csv", "test.csv", "sample_submission.csv"]
    if not all((DATA_DIR / name).exists() for name in required):
        archive = DATA_DIR / f"{COMPETITION}.zip"
        kaggle_executable = shutil.which("kaggle")
        if kaggle_executable is None:
            raise RuntimeError("Kaggle CLI executable not found after installation.")

        download_result = subprocess.run(
            [
                kaggle_executable,
                "competitions",
                "download",
                COMPETITION,
                "-p",
                str(DATA_DIR),
            ],
            check=False,
            text=True,
            capture_output=True,
        )
        if download_result.returncode != 0:
            raise RuntimeError(
                "Kaggle download failed.\n"
                f"stdout:\n{download_result.stdout}\n"
                f"stderr:\n{download_result.stderr}"
            )
        print(download_result.stdout)
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert TARGET in train and TARGET not in test
assert set(RAW_COLUMNS).issubset(train.columns)
assert list(sample_submission.columns) == [ID_COL, TARGET]
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert test[ID_COL].equals(sample_submission[ID_COL])
assert train[TARGET].isin([0, 1]).all()

if not SMOKE_TEST:
    assert len(train) == EXPECTED_TRAIN_ROWS
    assert len(test) == EXPECTED_TEST_ROWS
else:
    sampled, _ = train_test_split(
        train,
        train_size=min(60_000, len(train)),
        stratify=train[TARGET],
        random_state=FOLD_SEED,
    )
    train = sampled.sort_values(ID_COL).reset_index(drop=True)
    test = test.iloc[: min(20_000, len(test))].reset_index(drop=True)
    sample_submission = sample_submission.iloc[: len(test)].reset_index(drop=True)

y = train[TARGET].to_numpy(dtype=np.int8)
TRAIN_IDS = train[ID_COL].to_numpy()
TEST_IDS = test[ID_COL].to_numpy()

def vector_sha256(values):
    return hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()

DATA_SIGNATURE = hashlib.sha256(
    (
        vector_sha256(TRAIN_IDS)
        + vector_sha256(TEST_IDS)
        + str(len(train))
        + str(len(test))
    ).encode("utf-8")
).hexdigest()[:12]

print("train shape       ", train.shape)
print("test shape        ", test.shape)
print("positive rate     ", f"{y.mean():.6f}")
print("data signature    ", DATA_SIGNATURE)


## 5. Frozen folds and checkpoint contract

V7 uses seed 42 because this is the split exported by V5. A checkpoint is reused
only when its configuration, data fingerprint, validation indices, and prediction
shapes all match the current run.


In [ ]:
fold_ids = np.full(len(train), -1, dtype=np.int8)
splitter = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=FOLD_SEED,
)
for fold, (_, valid_idx) in enumerate(splitter.split(np.zeros(len(train)), y)):
    fold_ids[valid_idx] = fold
assert (fold_ids >= 0).all()
np.save(ARTIFACT_DIR / "fold_ids.npy", fold_ids)

def atomic_savez(path, **arrays):
    temporary = path.with_suffix(path.suffix + ".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    os.replace(temporary, path)

def checkpoint_path(member_name, fold):
    safe_name = member_name.replace("/", "_")
    return CHECKPOINT_DIR / f"{safe_name}_fold{fold}.npz"

def load_checkpoint(member_name, fold, expected_valid_idx):
    path = checkpoint_path(member_name, fold)
    if not path.exists():
        return None
    with np.load(path, allow_pickle=False) as saved:
        if str(saved["config_signature"].item()) != CONFIG_SIGNATURE:
            return None
        if str(saved["data_signature"].item()) != DATA_SIGNATURE:
            return None
        if not np.array_equal(saved["valid_idx"], expected_valid_idx):
            return None
        valid_pred = saved["valid_pred"].astype(np.float32)
        test_pred = saved["test_pred"].astype(np.float32)
        if valid_pred.shape != (len(expected_valid_idx),):
            return None
        if test_pred.shape != (len(test),):
            return None
        if not np.isfinite(valid_pred).all() or not np.isfinite(test_pred).all():
            return None
        return {
            "valid_pred": valid_pred,
            "test_pred": test_pred,
            "auc": float(saved["auc"].item()),
            "fit_seconds": float(saved["fit_seconds"].item()),
        }

def save_checkpoint(member_name, fold, valid_idx, valid_pred, test_pred, auc, seconds):
    atomic_savez(
        checkpoint_path(member_name, fold),
        config_signature=np.asarray(CONFIG_SIGNATURE),
        data_signature=np.asarray(DATA_SIGNATURE),
        valid_idx=np.asarray(valid_idx, dtype=np.int64),
        valid_pred=np.asarray(valid_pred, dtype=np.float32),
        test_pred=np.asarray(test_pred, dtype=np.float32),
        auc=np.asarray(auc, dtype=np.float64),
        fit_seconds=np.asarray(seconds, dtype=np.float64),
    )

print("fold counts", np.bincount(fold_ids))


## 6. Import aligned V5 and V6 OOF artifacts

This is not an import of leaderboard labels. These are self-trained, row-aligned
OOF and test predictions created by the previous notebooks. IDs, folds, target
values, shapes, and finite values are checked before registration.


In [ ]:
members = {}

def add_member(name, oof, test_prediction, family, source):
    oof = np.asarray(oof, dtype=np.float32)
    test_prediction = np.asarray(test_prediction, dtype=np.float32)
    assert oof.shape == (len(train),), f"OOF shape mismatch for {name}"
    assert test_prediction.shape == (len(test),), f"test shape mismatch for {name}"
    assert np.isfinite(oof).all() and np.isfinite(test_prediction).all()
    assert np.unique(test_prediction).size > 1000
    members[name] = {
        "oof": oof,
        "test": test_prediction,
        "family": family,
        "source": source,
    }

def newest_artifact(root, required_name):
    matches = [path.parent for path in root.glob(f"*/{required_name}")]
    if not matches:
        raise FileNotFoundError(f"No {required_name} found below {root}")
    return max(matches, key=lambda path: (path / required_name).stat().st_mtime)

V5_BASELINE_NAME = None
if not SMOKE_TEST:
    v5_root = Path("/content/drive/MyDrive/s6e8_top3_research_v4/artifacts_v5")
    v5_dir = newest_artifact(v5_root, "run_summary.json")
    v5_summary = json.loads((v5_dir / "run_summary.json").read_text(encoding="utf-8"))
    v5_oof = pd.read_csv(v5_dir / "oof_candidate_predictions.csv")
    v5_test = pd.read_csv(v5_dir / "test_candidate_predictions.csv")

    assert v5_oof[ID_COL].to_numpy().tolist() == TRAIN_IDS.tolist()
    assert v5_test[ID_COL].to_numpy().tolist() == TEST_IDS.tolist()
    assert np.array_equal(v5_oof[TARGET].to_numpy(dtype=np.int8), y)
    assert np.array_equal(v5_oof["fold"].to_numpy(dtype=np.int8), fold_ids)

    for short_name in V5_CANDIDATE_ALLOWLIST:
        column = f"candidate__{short_name}"
        if column in v5_oof and column in v5_test:
            add_member(
                f"v5__{short_name}",
                v5_oof[column],
                v5_test[column],
                family="v5_stack",
                source=str(v5_dir),
            )

    selected_short = v5_summary.get("selected_candidate")
    proposed_baseline = f"v5__{selected_short}"
    if proposed_baseline in members:
        V5_BASELINE_NAME = proposed_baseline
    else:
        v5_names = [name for name in members if name.startswith("v5__")]
        V5_BASELINE_NAME = max(
            v5_names,
            key=lambda name: roc_auc_score(y, members[name]["oof"]),
        )

    v6_root = Path("/content/drive/MyDrive/s6e8_v6_e1/artifacts")
    v6_dir = newest_artifact(v6_root, "run_manifest.json")
    add_member(
        "e1__maxbin1024",
        np.load(v6_dir / "oof_e1_maxbin1024.npy"),
        np.load(v6_dir / "test_e1_maxbin1024.npy"),
        family="xgb_raw_highbin",
        source=str(v6_dir),
    )

    print("V5 artifact", v5_dir)
    print("V6 artifact", v6_dir)
    print("V5 reference", V5_BASELINE_NAME)
else:
    print("SMOKE_TEST: previous full-run artifacts are intentionally skipped.")

print("registered imported members", sorted(members))


## 7. Target-free structural features

RealMLP receives raw features plus a compact set of behavioral identities. It does
not receive missing-count features because the competition has measurable
train/test missingness drift. Numerical medians are fitted separately inside each
outer training fold.


In [ ]:
def build_structural_features(frame):
    output = frame[RAW_COLUMNS].copy()
    output["allocated_screen"] = (
        output["social_media_hours"]
        + output["gaming_hours"]
        + output["work_study_hours"]
    )
    output["other_screen"] = (
        output["daily_screen_time_hours"] - output["allocated_screen"]
    )
    output["weekend_delta"] = (
        output["weekend_screen_time"] - output["daily_screen_time_hours"]
    )
    output["screen_to_sleep"] = (
        output["daily_screen_time_hours"] / output["sleep_hours"].clip(lower=0.25)
    )
    output["notifications_per_open"] = (
        output["notifications_per_day"] / output["app_opens_per_day"].clip(lower=1.0)
    )
    output["opens_per_screen_hour"] = (
        output["app_opens_per_day"]
        / output["daily_screen_time_hours"].clip(lower=0.25)
    )
    output["social_share"] = (
        output["social_media_hours"] / output["allocated_screen"].clip(lower=0.25)
    )
    output["gaming_share"] = (
        output["gaming_hours"] / output["allocated_screen"].clip(lower=0.25)
    )
    return output

realmlp_train_base = build_structural_features(train)
realmlp_test_base = build_structural_features(test)
REALMLP_CAT_COLUMNS = list(RAW_CATEGORICAL)
REALMLP_NUM_COLUMNS = [
    column for column in realmlp_train_base.columns
    if column not in REALMLP_CAT_COLUMNS
]

def prepare_realmlp_fold(train_idx, valid_idx):
    fold_train = realmlp_train_base.iloc[train_idx].copy()
    fold_valid = realmlp_train_base.iloc[valid_idx].copy()
    fold_test = realmlp_test_base.copy()

    medians = fold_train[REALMLP_NUM_COLUMNS].median()
    for frame in (fold_train, fold_valid, fold_test):
        frame[REALMLP_NUM_COLUMNS] = (
            frame[REALMLP_NUM_COLUMNS].fillna(medians).astype(np.float32)
        )
        for column in REALMLP_CAT_COLUMNS:
            frame[column] = frame[column].astype("string").fillna("__MISSING__")
    return fold_train, fold_valid, fold_test

print("RealMLP numeric features", len(REALMLP_NUM_COLUMNS))
print("RealMLP categorical features", REALMLP_CAT_COLUMNS)


## 8. Train the RealMLP OOF member

`n_refit=1` is important: RealMLP uses an internal split drawn only from the outer
training rows to estimate a useful epoch, then refits on all outer-training rows.
The outer validation labels never select the epoch.


In [ ]:
training_records = []

if RUN_REALMLP:
    member_name = "realmlp__target_free"
    realmlp_oof = np.full(len(train), np.nan, dtype=np.float32)
    realmlp_test_folds = []

    for fold in range(N_SPLITS):
        valid_idx = np.flatnonzero(fold_ids == fold)
        train_idx = np.flatnonzero(fold_ids != fold)
        cached = load_checkpoint(member_name, fold, valid_idx)

        if cached is None:
            fold_train, fold_valid, fold_test = prepare_realmlp_fold(
                train_idx, valid_idx
            )
            started = time.time()
            model = RealMLP_TD_Classifier(
                device=DEVICE,
                random_state=FOLD_SEED + 100 + fold,
                n_cv=1,
                n_refit=1,
                val_fraction=REALMLP_VAL_FRACTION,
                n_epochs=REALMLP_EPOCHS,
                batch_size=REALMLP_BATCH_SIZE,
                predict_batch_size=4096,
                val_metric_name="cross_entropy",
                use_ls=False,
                n_threads=2,
                verbosity=1,
            )
            model.fit(
                fold_train,
                y[train_idx],
                cat_col_names=REALMLP_CAT_COLUMNS,
            )
            valid_pred = model.predict_proba(fold_valid)[:, 1]
            test_pred = model.predict_proba(fold_test)[:, 1]
            seconds = time.time() - started
            auc = roc_auc_score(y[valid_idx], valid_pred)
            save_checkpoint(
                member_name,
                fold,
                valid_idx,
                valid_pred,
                test_pred,
                auc,
                seconds,
            )
            source = "trained"
            try:
                model.to("cpu")
            except Exception:
                pass
            del model, fold_train, fold_valid, fold_test
            gc.collect()
            if GPU_ACTIVE:
                torch.cuda.empty_cache()
        else:
            valid_pred = cached["valid_pred"]
            test_pred = cached["test_pred"]
            auc = cached["auc"]
            seconds = cached["fit_seconds"]
            source = "cache"

        realmlp_oof[valid_idx] = valid_pred
        realmlp_test_folds.append(np.asarray(test_pred, dtype=np.float32))
        training_records.append({
            "member": member_name,
            "fold": fold,
            "auc": auc,
            "seconds": seconds,
            "source": source,
        })
        print(f"RealMLP fold {fold}: AUC={auc:.7f} {source} {seconds/60:.1f} min")

    assert np.isfinite(realmlp_oof).all()
    add_member(
        member_name,
        realmlp_oof,
        np.mean(realmlp_test_folds, axis=0),
        family="realmlp",
        source="self-trained V7",
    )
    print("RealMLP OOF AUC", f"{roc_auc_score(y, realmlp_oof):.7f}")
else:
    print("RealMLP disabled")

del realmlp_train_base, realmlp_test_base
gc.collect()


## 9. Build CatBoost lattice features

Every raw numerical value is retained as a numerical feature. A second categorical
copy is added only after target-free rare-level bucketing on the provided train and
test features. CatBoost can then learn ordered target statistics for repeated grid
values without leaking outer validation labels.


In [ ]:
combined_raw = pd.concat(
    [train[RAW_COLUMNS], test[RAW_COLUMNS]],
    axis=0,
    ignore_index=True,
)
catboost_all = pd.DataFrame(index=np.arange(len(combined_raw)))

for column in RAW_NUMERIC:
    catboost_all[column] = pd.to_numeric(
        combined_raw[column], errors="coerce"
    ).astype(np.float32)

catboost_all["other_screen"] = (
    catboost_all["daily_screen_time_hours"]
    - catboost_all["social_media_hours"]
    - catboost_all["gaming_hours"]
    - catboost_all["work_study_hours"]
).astype(np.float32)

CATBOOST_CAT_COLUMNS = []
for column in RAW_CATEGORICAL:
    values = combined_raw[column].astype("string").fillna("__MISSING__")
    catboost_all[column] = pd.Categorical(values)
    CATBOOST_CAT_COLUMNS.append(column)

for column in RAW_NUMERIC:
    exact_name = f"{column}__exact_cat"
    values = combined_raw[column].astype("string").fillna("__MISSING__")
    frequencies = values.value_counts(dropna=False)
    retained = set(frequencies[frequencies >= CATBOOST_RARE_MIN_COUNT].index)
    bucketed = values.where(values.isin(retained), "__RARE__")
    catboost_all[exact_name] = pd.Categorical(bucketed)
    CATBOOST_CAT_COLUMNS.append(exact_name)

catboost_train = catboost_all.iloc[: len(train)].reset_index(drop=True)
catboost_test = catboost_all.iloc[len(train):].reset_index(drop=True)
del combined_raw, catboost_all
gc.collect()

print("CatBoost features", catboost_train.shape[1])
print("categorical copies", len(CATBOOST_CAT_COLUMNS))
print("RAM MB", f"{catboost_train.memory_usage(deep=True).sum()/2**20:.1f}")


## 10. Train the CatBoost lattice OOF member

Early stopping uses an inner split carved from the outer-training rows. The probe
model is discarded, and the selected iteration count is refitted on the complete
outer-training fold before predicting the held-out rows.


In [ ]:
def catboost_base_params(fold):
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "depth": 7,
        "learning_rate": 0.03,
        "l2_leaf_reg": 7.0,
        "random_strength": 0.7,
        "border_count": 254,
        "one_hot_max_size": 4,
        "random_seed": FOLD_SEED + 500 + fold,
        "allow_writing_files": False,
        "thread_count": 2,
        "verbose": False,
        "task_type": "GPU" if GPU_ACTIVE else "CPU",
    }
    if GPU_ACTIVE:
        params["devices"] = "0"
    return params

if RUN_CATBOOST_LATTICE:
    member_name = "catboost__lattice"
    cat_oof = np.full(len(train), np.nan, dtype=np.float32)
    cat_test_folds = []

    for fold in range(N_SPLITS):
        valid_idx = np.flatnonzero(fold_ids == fold)
        outer_train_idx = np.flatnonzero(fold_ids != fold)
        cached = load_checkpoint(member_name, fold, valid_idx)

        if cached is None:
            inner_train_idx, inner_valid_idx = train_test_split(
                outer_train_idx,
                test_size=CATBOOST_INNER_VALID_FRACTION,
                stratify=y[outer_train_idx],
                random_state=FOLD_SEED + 700 + fold,
            )
            started = time.time()
            params = catboost_base_params(fold)
            probe = CatBoostClassifier(
                iterations=CATBOOST_MAX_ITERATIONS,
                od_type="Iter",
                od_wait=CATBOOST_EARLY_STOPPING,
                use_best_model=True,
                **params,
            )
            probe.fit(
                Pool(
                    catboost_train.iloc[inner_train_idx],
                    y[inner_train_idx],
                    cat_features=CATBOOST_CAT_COLUMNS,
                ),
                eval_set=Pool(
                    catboost_train.iloc[inner_valid_idx],
                    y[inner_valid_idx],
                    cat_features=CATBOOST_CAT_COLUMNS,
                ),
                verbose=250,
            )
            best_iterations = max(50, int(probe.get_best_iteration()) + 1)
            del probe
            gc.collect()

            final_model = CatBoostClassifier(
                iterations=best_iterations,
                use_best_model=False,
                **params,
            )
            final_model.fit(
                Pool(
                    catboost_train.iloc[outer_train_idx],
                    y[outer_train_idx],
                    cat_features=CATBOOST_CAT_COLUMNS,
                ),
                verbose=250,
            )
            valid_pred = final_model.predict_proba(
                catboost_train.iloc[valid_idx]
            )[:, 1]
            test_pred = final_model.predict_proba(catboost_test)[:, 1]
            seconds = time.time() - started
            auc = roc_auc_score(y[valid_idx], valid_pred)
            save_checkpoint(
                member_name,
                fold,
                valid_idx,
                valid_pred,
                test_pred,
                auc,
                seconds,
            )
            source = "trained"
            del final_model
            gc.collect()
        else:
            valid_pred = cached["valid_pred"]
            test_pred = cached["test_pred"]
            auc = cached["auc"]
            seconds = cached["fit_seconds"]
            source = "cache"

        cat_oof[valid_idx] = valid_pred
        cat_test_folds.append(np.asarray(test_pred, dtype=np.float32))
        training_records.append({
            "member": member_name,
            "fold": fold,
            "auc": auc,
            "seconds": seconds,
            "source": source,
        })
        print(
            f"CatBoost lattice fold {fold}: AUC={auc:.7f} "
            f"{source} {seconds/60:.1f} min"
        )

    assert np.isfinite(cat_oof).all()
    add_member(
        member_name,
        cat_oof,
        np.mean(cat_test_folds, axis=0),
        family="catboost_lattice",
        source="self-trained V7",
    )
    print("CatBoost lattice OOF AUC", f"{roc_auc_score(y, cat_oof):.7f}")
else:
    print("CatBoost lattice disabled")

del catboost_train, catboost_test
gc.collect()


## 11. Member strength and diversity audit

A member is not valuable merely because its standalone AUC is high. This section
measures both AUC and rank correlation before the meta-model is allowed to use it.


In [ ]:
if len(members) < 2:
    raise RuntimeError("V7 needs at least two completed members for stacking.")

member_names = list(members)
member_oof = np.column_stack([members[name]["oof"] for name in member_names])
member_test = np.column_stack([members[name]["test"] for name in member_names])

member_scores = pd.DataFrame([
    {
        "member": name,
        "family": members[name]["family"],
        "oof_auc": roc_auc_score(y, members[name]["oof"]),
    }
    for name in member_names
]).sort_values("oof_auc", ascending=False).reset_index(drop=True)
display(member_scores)

rank_oof = np.column_stack([
    rankdata(member_oof[:, column], method="average").astype(np.float32)
    for column in range(member_oof.shape[1])
])
rank_correlation = pd.DataFrame(
    np.corrcoef(rank_oof, rowvar=False),
    index=member_names,
    columns=member_names,
)
display(rank_correlation.round(5))

member_scores.to_csv(ARTIFACT_DIR / "member_scores.csv", index=False)
rank_correlation.to_csv(ARTIFACT_DIR / "member_oof_spearman.csv")
pd.DataFrame(training_records).to_csv(
    ARTIFACT_DIR / "new_member_training_records.csv", index=False
)


## 12. Nested correlation-clustered logit stacking

For each held-out fold, member clusters and representatives are selected using
only the other folds. The L2 value is then chosen by an inner CV loop. Fold-bagged
test predictions come from the same five meta-models that create the nested OOF
estimate.


In [ ]:
def clipped_logit(values, epsilon=1e-6):
    clipped = np.clip(values, epsilon, 1.0 - epsilon)
    return np.log(clipped / (1.0 - clipped)).astype(np.float32)

def representatives_for_rows(names, row_idx):
    if len(names) <= 1:
        return names, {names[0]: 1}
    matrix = np.column_stack([
        rankdata(members[name]["oof"][row_idx], method="average")
        for name in names
    ]).astype(np.float32)
    correlation = np.corrcoef(matrix, rowvar=False)
    distance = squareform(
        np.clip(1.0 - correlation, 0.0, 2.0), checks=False
    )
    labels = fcluster(
        linkage(distance, method="average"),
        t=1.0 - CORRELATION_CLUSTER_THRESHOLD,
        criterion="distance",
    )
    selected = []
    label_map = {}
    for label in np.unique(labels):
        positions = np.flatnonzero(labels == label)
        best_position = max(
            positions,
            key=lambda position: roc_auc_score(
                y[row_idx], members[names[position]]["oof"][row_idx]
            ),
        )
        selected.append(names[best_position])
        for position in positions:
            label_map[names[position]] = int(label)
    return selected, label_map

def matrix_for(names, rows, source):
    if source == "oof":
        return np.column_stack([
            clipped_logit(members[name]["oof"][rows]) for name in names
        ])
    if source == "test":
        return np.column_stack([
            clipped_logit(members[name]["test"]) for name in names
        ])
    raise ValueError(source)

def select_meta_c(names, train_idx, fold):
    inner = StratifiedKFold(
        n_splits=4 if not SMOKE_TEST else 3,
        shuffle=True,
        random_state=FOLD_SEED + 2000 + fold,
    )
    scores = {value: [] for value in META_C_GRID}
    local_y = y[train_idx]
    for inner_train_local, inner_valid_local in inner.split(
        np.zeros(len(train_idx)), local_y
    ):
        inner_train_idx = train_idx[inner_train_local]
        inner_valid_idx = train_idx[inner_valid_local]
        x_train = matrix_for(names, inner_train_idx, "oof")
        x_valid = matrix_for(names, inner_valid_idx, "oof")
        for value in META_C_GRID:
            model = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    C=value,
                    penalty="l2",
                    solver="lbfgs",
                    max_iter=1000,
                    random_state=FOLD_SEED + fold,
                ),
            )
            model.fit(x_train, y[inner_train_idx])
            prediction = model.predict_proba(x_valid)[:, 1]
            scores[value].append(roc_auc_score(y[inner_valid_idx], prediction))
    means = {value: float(np.mean(values)) for value, values in scores.items()}
    best_value = max(META_C_GRID, key=lambda value: (means[value], -value))
    return best_value, means

meta_oof = np.full(len(train), np.nan, dtype=np.float32)
meta_test_folds = []
meta_fold_records = []
coefficient_rows = []

eligible_names = [
    name for name in member_names
    if roc_auc_score(y, members[name]["oof"]) >= 0.955
]

for fold in range(N_SPLITS):
    valid_idx = np.flatnonzero(fold_ids == fold)
    train_idx = np.flatnonzero(fold_ids != fold)
    selected_names, cluster_map = representatives_for_rows(
        eligible_names, train_idx
    )
    selected_c, c_scores = select_meta_c(selected_names, train_idx, fold)

    x_train = matrix_for(selected_names, train_idx, "oof")
    x_valid = matrix_for(selected_names, valid_idx, "oof")
    x_test = matrix_for(selected_names, np.arange(len(test)), "test")
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=selected_c,
            penalty="l2",
            solver="lbfgs",
            max_iter=1500,
            random_state=FOLD_SEED + 3000 + fold,
        ),
    )
    model.fit(x_train, y[train_idx])
    meta_oof[valid_idx] = model.predict_proba(x_valid)[:, 1]
    meta_test_folds.append(model.predict_proba(x_test)[:, 1])

    fold_auc = roc_auc_score(y[valid_idx], meta_oof[valid_idx])
    meta_fold_records.append({
        "fold": fold,
        "auc": fold_auc,
        "selected_c": selected_c,
        "members": " | ".join(selected_names),
        "inner_c_scores": json.dumps(c_scores),
    })
    coefficients = model.named_steps["logisticregression"].coef_[0]
    for name, value in zip(selected_names, coefficients):
        coefficient_rows.append({
            "fold": fold,
            "member": name,
            "coefficient": float(value),
            "cluster": cluster_map[name],
            "selected_c": selected_c,
        })
    print(
        f"meta fold {fold}: AUC={fold_auc:.7f} C={selected_c} "
        f"members={selected_names}"
    )
    del x_train, x_valid, x_test, model
    gc.collect()

assert np.isfinite(meta_oof).all()
meta_test = np.mean(meta_test_folds, axis=0).astype(np.float32)
add_member(
    "v7__nested_logit_stack",
    meta_oof,
    meta_test,
    family="v7_meta",
    source="nested fold-bagged V7",
)

print("V7 nested OOF AUC", f"{roc_auc_score(y, meta_oof):.7f}")
display(pd.DataFrame(meta_fold_records))
coefficient_frame = pd.DataFrame(coefficient_rows)
display(coefficient_frame.pivot_table(
    index="member", columns="fold", values="coefficient"
))

pd.DataFrame(meta_fold_records).to_csv(
    ARTIFACT_DIR / "meta_fold_records.csv", index=False
)
coefficient_frame.to_csv(
    ARTIFACT_DIR / "meta_coefficients.csv", index=False
)


## 13. Frozen acceptance gate

The V7 stack is compared with the V5 selected reference, not with the public
leaderboard. The gate requires an overall gain, at least four fold wins, and no
fold loss larger than the precommitted tolerance.


In [ ]:
if V5_BASELINE_NAME is None:
    non_meta = [name for name in members if name != "v7__nested_logit_stack"]
    V5_BASELINE_NAME = max(
        non_meta,
        key=lambda name: roc_auc_score(y, members[name]["oof"]),
    )

reference_oof = members[V5_BASELINE_NAME]["oof"]
stack_oof = members["v7__nested_logit_stack"]["oof"]
reference_auc = roc_auc_score(y, reference_oof)
stack_auc = roc_auc_score(y, stack_oof)
overall_gain = stack_auc - reference_auc

fold_rows = []
for fold in range(N_SPLITS):
    mask = fold_ids == fold
    reference_fold_auc = roc_auc_score(y[mask], reference_oof[mask])
    stack_fold_auc = roc_auc_score(y[mask], stack_oof[mask])
    fold_rows.append({
        "fold": fold,
        "reference_auc": reference_fold_auc,
        "v7_auc": stack_fold_auc,
        "delta": stack_fold_auc - reference_fold_auc,
    })
fold_comparison = pd.DataFrame(fold_rows)
display(fold_comparison)

fold_wins = int((fold_comparison["delta"] > 0).sum())
worst_fold_delta = float(fold_comparison["delta"].min())
V7_ACCEPTED = bool(
    overall_gain > MIN_OVERALL_GAIN
    and fold_wins >= MIN_FOLD_WINS
    and worst_fold_delta >= -MAX_WORST_FOLD_LOSS
)

print(f"reference                   {V5_BASELINE_NAME}")
print(f"reference OOF AUC           {reference_auc:.7f}")
print(f"V7 stack OOF AUC            {stack_auc:.7f}")
print(f"overall gain                {overall_gain:+.7f}")
print(f"fold wins                   {fold_wins}/{N_SPLITS}")
print(f"worst fold delta            {worst_fold_delta:+.7f}")
print(f"V7 ACCEPTED                 {V7_ACCEPTED}")

SELECTED_NAME = "v7__nested_logit_stack" if V7_ACCEPTED else V5_BASELINE_NAME
print("selected self-trained prediction", SELECTED_NAME)

fold_comparison.to_csv(ARTIFACT_DIR / "v7_acceptance_folds.csv", index=False)


## 14. Anchor and public-transfer screening

This section measures test-rank correlation with the existing `0.97128` anchor.
The public score of V7 is unknown, so `OOF + 0.00102` is displayed only as a prior
calibrated from the observed V5 and E1 transfers. It is not treated as truth.


In [ ]:
anchor_candidates = [
    Path("/content/drive/MyDrive/s6e8_v6_e1/anchors/submission_s6e8_community_blend.csv"),
    Path("/content/drive/MyDrive/s6e8_top3_research_v4/anchors/submission_s6e8_community_blend.csv"),
]
ANCHOR_PATH = next((path for path in anchor_candidates if path.exists()), None)

def d_from_auc(value):
    return math.sqrt(2.0) * norm.ppf(value)

def auc_from_d(value):
    return float(norm.cdf(value / math.sqrt(2.0)))

def unconstrained_weight(d_member, d_anchor, rho):
    return (d_member - rho * d_anchor) / (
        (1.0 - rho) * (d_anchor + d_member)
    )

anchor_screen_rows = []
ANCHOR_AVAILABLE = ANCHOR_PATH is not None
anchor = None

if ANCHOR_AVAILABLE:
    anchor = pd.read_csv(ANCHOR_PATH)
    assert list(anchor.columns) == [ID_COL, TARGET]
    assert anchor[ID_COL].to_numpy().tolist() == TEST_IDS.tolist()
    assert np.isfinite(anchor[TARGET]).all()
    anchor_values = anchor[TARGET].to_numpy(dtype=np.float64)
    d_anchor = d_from_auc(ANCHOR_PUBLIC_AUC)

    screen_names = [
        name for name in [
            "realmlp__target_free",
            "catboost__lattice",
            "v7__nested_logit_stack",
        ]
        if name in members
    ]
    for name in screen_names:
        oof_auc = roc_auc_score(y, members[name]["oof"])
        assumed_public = min(oof_auc + OOF_TO_PUBLIC_PRIOR, 0.999999)
        rho = float(spearmanr(members[name]["test"], anchor_values).statistic)
        break_even_auc = auc_from_d(rho * d_anchor)
        raw_weight = unconstrained_weight(
            d_from_auc(assumed_public), d_anchor, rho
        )
        anchor_screen_rows.append({
            "member": name,
            "oof_auc": oof_auc,
            "assumed_public_auc": assumed_public,
            "rho_vs_anchor": rho,
            "break_even_auc": break_even_auc,
            "clears_break_even": assumed_public > break_even_auc,
            "w_star_unconstrained": raw_weight,
            "w_star_clipped": float(np.clip(raw_weight, 0.0, 1.0)),
        })

    anchor_screen = pd.DataFrame(anchor_screen_rows)
    display(anchor_screen)
    anchor_screen.to_csv(ARTIFACT_DIR / "anchor_screen.csv", index=False)
else:
    anchor_screen = pd.DataFrame()
    print("Anchor not found. The self-trained submissions are still valid.")


## 15. Export auditable submissions

The notebook exports the two new standalone members and the V7 stack. This does
not mean all three should be submitted. Read the final recommendation first.


In [ ]:
def build_submission(prediction, filename):
    if SMOKE_TEST:
        raise RuntimeError("SMOKE_TEST outputs cannot be exported as submissions.")
    prediction = np.asarray(prediction, dtype=np.float64)
    assert prediction.shape == (len(test),)
    assert np.isfinite(prediction).all()
    frame = pd.DataFrame({ID_COL: TEST_IDS, TARGET: prediction})
    assert frame[ID_COL].to_numpy().tolist() == TEST_IDS.tolist()
    assert frame[TARGET].nunique() > 1000
    path = SUBMISSION_DIR / filename
    frame.to_csv(path, index=False)
    local_path = Path("/content") / filename
    if IN_COLAB:
        local_path.write_bytes(path.read_bytes())
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(filename)
    print("  rows     ", f"{len(frame):,}")
    print("  range    ", f"[{prediction.min():.8f}, {prediction.max():.8f}]")
    print("  distinct ", f"{frame[TARGET].nunique():,}")
    print("  sha256   ", digest)
    print("  drive    ", path)
    return path, digest

submission_hashes = {}
submission_paths = {}

if SMOKE_TEST:
    print("SMOKE_TEST active: no submission files created.")
else:
    export_map = {
        "v7_self_stack": "v7__nested_logit_stack",
    }
    if "realmlp__target_free" in members:
        export_map["v7_realmlp"] = "realmlp__target_free"
    if "catboost__lattice" in members:
        export_map["v7_catboost_lattice"] = "catboost__lattice"

    for file_tag, member_name in export_map.items():
        filename = f"submission_s6e8_{file_tag}.csv"
        path, digest = build_submission(members[member_name]["test"], filename)
        submission_paths[file_tag] = str(path)
        submission_hashes[filename] = digest
        print()


## 16. Optional anchor blend, disabled by default

Enable this only after the selected member clears the section 14 break-even test.
The default notebook never creates a blend merely because the public anchor is
stronger.


In [ ]:
ENABLE_ANCHOR_BLEND = False
BLEND_MEMBER = "v7__nested_logit_stack"
BLEND_WEIGHT = None

if not ENABLE_ANCHOR_BLEND:
    print("Anchor blend disabled.")
else:
    if not ANCHOR_AVAILABLE:
        raise RuntimeError("The anchor file is required.")
    row = anchor_screen.loc[anchor_screen["member"] == BLEND_MEMBER]
    if row.empty or not bool(row.iloc[0]["clears_break_even"]):
        raise RuntimeError("The selected member does not clear the break-even screen.")
    if BLEND_WEIGHT is None:
        raise ValueError("Set BLEND_WEIGHT explicitly after reading section 14.")
    if not 0.0 < BLEND_WEIGHT < 1.0:
        raise ValueError("BLEND_WEIGHT must be strictly between zero and one.")

    member_rank = (rankdata(members[BLEND_MEMBER]["test"]) - 0.5) / len(test)
    anchor_rank = (rankdata(anchor[TARGET].to_numpy()) - 0.5) / len(test)
    blended = BLEND_WEIGHT * member_rank + (1.0 - BLEND_WEIGHT) * anchor_rank
    tag = int(round(100 * BLEND_WEIGHT))
    filename = f"submission_s6e8_v7_anchor_self{tag:02d}.csv"
    path, digest = build_submission(blended, filename)
    submission_paths["anchor_blend"] = str(path)
    submission_hashes[filename] = digest


## 17. Run manifest and decision summary

A leaderboard target is not a validation criterion. The manifest records the
evidence that existed before any V7 file was uploaded.


In [ ]:
manifest = {
    "pipeline": PIPELINE_VERSION,
    "config_signature": CONFIG_SIGNATURE,
    "data_signature": DATA_SIGNATURE,
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "pytabkit": PYTABKIT_VERSION,
        "catboost": CATBOOST_VERSION,
        "gpu_active": GPU_ACTIVE,
        "device": DEVICE,
    },
    "config": CONFIG,
    "member_scores": member_scores.to_dict(orient="records"),
    "reference_member": V5_BASELINE_NAME,
    "reference_oof_auc": reference_auc,
    "v7_oof_auc": stack_auc,
    "v7_gain": overall_gain,
    "v7_fold_wins": fold_wins,
    "v7_worst_fold_delta": worst_fold_delta,
    "v7_accepted": V7_ACCEPTED,
    "selected_self_trained_member": SELECTED_NAME,
    "anchor_public_auc_reference": ANCHOR_PUBLIC_AUC,
    "dated_public_leader_reference": CURRENT_PUBLIC_TARGET,
    "anchor_screen": anchor_screen_rows,
    "submission_paths": submission_paths,
    "submission_sha256": submission_hashes,
}
manifest_path = ARTIFACT_DIR / "run_manifest.json"
temporary_manifest = manifest_path.with_suffix(".json.tmp")
temporary_manifest.write_text(
    json.dumps(manifest, indent=2, default=str), encoding="utf-8"
)
os.replace(temporary_manifest, manifest_path)

print(json.dumps({
    "reference": V5_BASELINE_NAME,
    "reference_oof_auc": reference_auc,
    "v7_oof_auc": stack_auc,
    "v7_gain": overall_gain,
    "fold_wins": fold_wins,
    "worst_fold_delta": worst_fold_delta,
    "v7_accepted": V7_ACCEPTED,
}, indent=2))
print()
print("artifacts", ARTIFACT_DIR)
for path in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  {path.name:<42} {path.stat().st_size / 1024:>10.1f} KB")


## 18. Submission policy

Use no more than three new public submissions:

1. Submit `submission_s6e8_v7_self_stack.csv` first **only when V7 ACCEPTED is
   True**. It is the primary scientific result.
2. Submit the strongest new standalone family only when it is sufficiently strong
   and genuinely decorrelated. Its purpose is to measure public transfer, not to
   search weights.
3. Keep the third slot in reserve for the single section 14 blend that passes the
   break-even screen. Do not submit a grid of nearby weights.

Keep the credited `0.97128` community anchor among the final candidates. A public
score above the current leader is possible but cannot be guaranteed, and a small
public improvement must still survive the private split.

## How to run on Colab

1. Select **Runtime -> Change runtime type -> T4 GPU**.
2. Confirm that V5 and V6 artifacts still exist in Google Drive.
3. Set `SMOKE_TEST=True`, run once, and confirm both new members finish.
4. Restart the runtime, restore `SMOKE_TEST=False`, and run the full notebook.
5. After a disconnect, run from the top. Completed folds load from Drive.
6. Read sections 11 through 14 before uploading any CSV.

## Portfolio interpretation

This notebook is valuable even when the gate rejects V7. A rejection shows that
additional model complexity did not produce stable marginal information after
clustering and nested stacking. That is a stronger engineering result than a
leaderboard-only blend with unknown generalization.
